In [ ]:
# ## console

# (async function() {
#     // 1. استخراج matchId تلقائياً من رابط الصفحة المفتوحة
#     let matchId = null;
#     const hashMatch = window.location.hash.match(/id:(\d+)/);
#     if (hashMatch) {
#         matchId = hashMatch[1];
#     } else {
#         const pathMatch = window.location.pathname.match(/\/(\d+)$/);
#         if (pathMatch) matchId = pathMatch[1];
#     }

#     if (!matchId) {
#         console.error("[-] لم يتم العثور على matchId تلقائياً من الرابط! تأكد أنك داخل صفحة الماتش.");
#         return;
#     }

#     console.log(`[+] جاري استخراج بيانات الماتش رقم: ${matchId}...`);
    
#     // 2. جلب البيانات من الـ API
#     const [resEvent, resLineups] = await Promise.all([
#         fetch(`/api/v1/event/${matchId}`).then(r => r.json()),
#         fetch(`/api/v1/event/${matchId}/lineups`).then(r => r.json())
#     ]);

#     const event = resEvent.event || {};
#     const homeTeam = event.homeTeam?.name || "Home";
#     const awayTeam = event.awayTeam?.name || "Away";
#     const homeScore = event.homeScore?.current || 0;
#     const awayScore = event.awayScore?.current || 0;

#     let rows = [];

#     function processPlayers(players, teamName, isHome) {
#         (players || []).forEach(p => {
#             if (!p.statistics) return;
#             const s = p.statistics;
#             const minutes = s.minutesPlayed || 0;
            
#             // حسابات الالتحامات المجمعة
#             const duelWon = s.duelWon || 0;
#             const duelLost = s.duelLost || 0;
#             const totalDuels = duelWon + duelLost;

#             const aerWon = s.aerialWon || s.aerialDuelWon || 0;
#             const aerLost = s.aerialLost || s.aerialDuelLost || 0;
#             const aerTotal = aerWon + aerLost;

#             const grdWon = Math.max(0, duelWon - aerWon);
#             const grdTotal = Math.max(0, totalDuels - aerTotal);

#             // الكلين شيت
#             const conceded = isHome ? awayScore : homeScore;
#             const cleanSheet = (conceded === 0 && minutes >= 60 && ['G', 'D'].includes(p.position)) ? 1 : 0;

#             // البيانات الأساسية للجدول والعرض
#             const baseData = {
#                 match_id: Number(matchId),
#                 player_id: p.player?.id || null,
#                 player_name: p.player?.name || "",
#                 team: teamName,
#                 position: p.position || "",
#                 minutes_played: minutes,
#                 sofascore_rating: s.rating || 0.0,
#                 clean_sheet: cleanSheet,
#                 display_passes: `${s.accuratePass || 0}/${s.totalPass || 0}`,
#                 display_total_duels: `${totalDuels} (${duelWon})`,
#                 display_ground_duels: `${grdTotal} (${grdWon})`,
#                 display_aerial_duels: `${aerTotal} (${aerWon})`,
#                 ground_duels_won: grdWon,
#                 ground_duels_total: grdTotal,
#                 aerial_duels_won: aerWon,
#                 aerial_duels_total: aerTotal
#             };

#             // دمج الإحصائيات الخام وحذف الكائنات المتداخلة غير الرقمية
#             const rawStats = { ...s };
#             delete rawStats.ratingVersions;
#             delete rawStats.statisticsType;

#             rows.push({
#                 ...baseData,
#                 ...rawStats
#             });
#         });
#     }

#     processPlayers(resLineups.home?.players, homeTeam, true);
#     processPlayers(resLineups.away?.players, awayTeam, false);

#     // 3. تنزيل الملف بالـ ID الخاص بالمباراة الحالية تلقائياً
#     const blob = new Blob([JSON.stringify(rows, null, 2)], { type: 'application/json' });
#     const url = URL.createObjectURL(blob);
#     const a = document.createElement('a');
#     a.href = url;
#     a.download = `match_${matchId}_full_stats.json`;
#     document.body.appendChild(a);
#     a.click();
#     document.body.removeChild(a);
#     console.log(`✓ تم بنجاح! تم حفظ ملف match_${matchId}_full_stats.json بعدد ${rows.length} لاعب.`);
# })();    

# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------
# -----------------------------------------------------------------------

# import glob
# import pandas as pd

# # قراءة كل ملفات الـ JSON في الفولدر الحالي
# json_files = glob.glob("match_*.json")
# if not json_files:
#     json_files = glob.glob("match_*")

# print(f"[+] جاري دمج {len(json_files)} ملفات شاملة كل اللاعبين...")

# dataframes = []
# for file in json_files:
#     try:
#         df_match = pd.read_json(file)
#         dataframes.append(df_match)
#     except Exception as e:
#         print(f"[-] خطأ في {file}: {e}")

# if dataframes:
#     # 1. دمج جميع اللاعبين بلا أي حذف أو استبعاد
#     df_all = pd.concat(dataframes, ignore_index=True)

#     # 2. ملء كل الخانات المفقودة في الأعمدة الرقمية بـ 0
#     numeric_cols = df_all.select_dtypes(include=['float64', 'int64']).columns
#     df_all[numeric_cols] = df_all[numeric_cols].fillna(0)

#     # 3. ملء الخانات النصية الفارغة بشرطة
#     df_all = df_all.fillna("-")

#     # 4. حفظ الملف المجمع الكامل
#     output_filename = "game_week_1.csv"
#     df_all.to_csv(output_filename, index=False, encoding="utf-8-sig")

#     print(f"[✓] تم الحفظ بنجاح بدون استبعاد أي لاعب!")
#     print(f"[✓] إجمالي عدد اللاعبين في الجولة (أساسي + دكة): {len(df_all)}")
#     print(f"[✓] اسم الملف: {output_filename}")
#     display(df_all.head())


In [2]:
%pip install pandas

     ---------------------------------------- 11.0/11.0 MB 3.6 MB/s eta 0:00:00
     -------------------------------------- 348.2/348.2 kB 3.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import glob
import pandas as pd

# قراءة كل ملفات الـ JSON في الفولدر الحالي
json_files = glob.glob("match_*.json")
if not json_files:
    json_files = glob.glob("match_*")

print(f"[+] جاري دمج {len(json_files)} ملفات شاملة كل اللاعبين...")

dataframes = []
for file in json_files:
    try:
        df_match = pd.read_json(file)
        dataframes.append(df_match)
    except Exception as e:
        print(f"[-] خطأ في {file}: {e}")

if dataframes:
    # 1. دمج جميع اللاعبين بلا أي حذف أو استبعاد
    df_all = pd.concat(dataframes, ignore_index=True)

    # 2. ملء كل الخانات المفقودة في الأعمدة الرقمية بـ 0
    numeric_cols = df_all.select_dtypes(include=['float64', 'int64']).columns
    df_all[numeric_cols] = df_all[numeric_cols].fillna(0)

    # 3. ملء الخانات النصية الفارغة بشرطة
    df_all = df_all.fillna("-")

    # 4. حفظ الملف المجمع الكامل
    output_filename = "game_week7.csv"
    df_all.to_csv(output_filename, index=False, encoding="utf-8-sig")

    print(f"[✓] تم الحفظ بنجاح بدون استبعاد أي لاعب!")
    print(f"[✓] إجمالي عدد اللاعبين في الجولة (أساسي + دكة): {len(df_all)}")
    print(f"[✓] اسم الملف: {output_filename}")
    display(df_all.head())

[+] جاري دمج 10 ملفات شاملة كل اللاعبين...
[✓] تم الحفظ بنجاح بدون استبعاد أي لاعب!
[✓] إجمالي عدد اللاعبين في الجولة (أساسي + دكة): 400
[✓] اسم الملف: game_week7.csv


,match_id,player_id,player_name,team,position,minutes_played,sofascore_rating,clean_sheet,display_passes,display_total_duels,...,passValueNormalized,dribbleValueNormalized,defensiveValueNormalized,goalkeeperValueNormalized,expectedGoals,expectedGoalsOnTarget,bestBallCarryProgression,totalProgressiveBallCarriesDistance,keyPass,shotValueNormalized
0,16738980,1481183,Mohamed Seha,Al-Masry,G,90,6.8,0,11/17,3 (3),...,-0.01,-0.02,-0.05,0.08,0.0000,0.0000,0.000000,0.000000,0.0,0.00
1,16738980,918727,Karim El Eraki,Al-Masry,D,90,6.6,0,24/42,7 (5),...,0.01,0.28,-0.06,0.00,0.1990,0.1651,23.398693,32.653536,1.0,-0.14
2,16738980,918604,Baher El Mohamady,Al-Masry,D,90,6.9,0,39/46,2 (2),...,0.34,0.07,-0.13,0.00,0.0331,0.0000,0.000000,0.000000,0.0,-0.13
3,16738980,1173530,Mostafa Mahmoud Al Aash,Al-Masry,D,90,7.1,0,53/62,8 (6),...,0.18,0.08,-0.08,0.00,0.0119,0.0000,0.000000,0.000000,0.0,-0.03
4,16738980,1050729,Mohamed Shokry,Al-Masry,D,89,7.0,0,39/49,10 (6),...,0.24,0.06,-0.02,0.00,0.0328,0.0000,0.000000,0.000000,2.0,-0.14


تاكيدات

In [2]:
print("عدد الماتشات:", df_all['match_id'].nunique())
print("عدد الفرق:", df_all['team'].nunique())
print("قائمة الفرق:\n", df_all['team'].unique())

عدد الماتشات: 10
عدد الفرق: 20
قائمة الفرق:
 <StringArray>
[            'Wadi Degla',                 'ZED FC',             'Zamalek SC',
  'Al-Ittihad Alexandria', 'Abu Qir Fertilizers SC', 'National Bank of Egypt',
            'El Gouna FC',        'Modern Sport FC',       'Tala'ea El-Gaish',
    'Arab Contractors FC',    'Ghazl El Mahalla FC',            'Pyramids FC',
           'Cleopatra FC',               'Canal SC',               'Al-Masry',
                 'Smouha',             'Al Ahly FC',    'El-Sharkia Enppi FC',
        'Asyut Petroleum',       'Suez Petrojet FC']
Length: 20, dtype: str


In [3]:
# فحص أقصى وأقل عدد دقائق
print("أقصى دقائق:", df_all['minutes_played'].max())
print("أقل دقائق:", df_all['minutes_played'].min())

# عدد اللاعبين الأساسيين والبدلاء
starters = df_all[df_all['minutes_played'] >= 45]
bench_unplayed = df_all[df_all['minutes_played'] == 0]

print(f"عدد اللي لعبوا 45 دقيقة فأكثر: {len(starters)}")
print(f"عدد اللي قعدوا دكة بدون مشاركة: {len(bench_unplayed)}")

أقصى دقائق: 90
أقل دقائق: 0
عدد اللي لعبوا 45 دقيقة فأكثر: 231
عدد اللي قعدوا دكة بدون مشاركة: 85


In [4]:
# اعرض هدافي الجولة بالكامل للتأكد من الأسماء
top_scorers = df_all[df_all['goals'] > 0][['player_name', 'team', 'goals', 'minutes_played']]
print(top_scorers)

             player_name                    team  goals  minutes_played
93           Ali Hussein  Abu Qir Fertilizers SC    1.0              32
134           Ahmed Zaki             El Gouna FC    1.0               8
150      Mahmoud Mamdouh         Modern Sport FC    1.0              90
161     Ahmed Alaa Eldin        Tala'ea El-Gaish    1.0              90
227        Odeh Fakhoury             Pyramids FC    1.0              66
229       Mahmoud Zalaka             Pyramids FC    1.0              76
231         Nasser Maher             Pyramids FC    1.0              24
248         Mohamed Reda            Cleopatra FC    1.0              90
262         Ibrahim Samy                Canal SC    1.0              90
272          Mawuli Wayo                Canal SC    1.0              29
327         Marwan Attia              Al Ahly FC    1.0              90
330                 Zizo              Al Ahly FC    1.0              86
334  Mohamed Magdy Afsha              Al Ahly FC    1.0         

In [5]:
duplicates = df_all.duplicated(subset=['match_id', 'player_id']).sum()
print("عدد السجلات المكررة:", duplicates) # المفروض يطلع 0

عدد السجلات المكررة: 0


In [ ]:
print(df_all['display_passes'].head(30))

0     17/20
1     43/47
2     67/73
3     73/81
4     18/25
5     28/35
6     13/19
7     11/18
8     10/14
9     15/17
10      4/5
11      7/7
12      3/4
13      3/4
14      2/2
15      0/0
16      0/0
17      0/0
18      0/0
19      0/0
20    18/27
21    16/21
22    11/20
23    24/32
24    20/27
25    23/28
26    12/17
27    12/17
28    11/14
29    13/18
Name: display_passes, dtype: str


: 